# MongoDB Integration

In this notebook, the processed parquet output generated by the Spark processing notebook is loaded into MongoDB.
The goal is to store the processed order-level data in MongoDB, create indexes, and run example queries for analysis.

# 01 Import Libraries

In this section, we import the Python libraries needed for MongoDB connection, 
data handling, and performance testing.

In [ ]:
from pymongo import MongoClient
import pandas as pd
import os
import time

# 02 Connect to MongoDB

Here, we create a connection to the local MongoDB server and define the database 
and collection that will be used in this project.

In [ ]:
client = MongoClient("mongodb://localhost:27017/")
db = client["olist_ecommerce"]
collection = db["orders_simple"]

print("Connected to MongoDB")

Connected to MongoDB


# 03 Load Processed Data

In this section, we load the processed parquet outputs generated by the Spark processing notebook.
This allows the MongoDB notebook to continue from the previous pipeline stage instead of reading the raw CSV files again.

In [ ]:
processed_path = "../data/processed/orders_core.parquet"

print("Current working directory:", os.getcwd())
print("Processed file exists:", os.path.exists(processed_path))

df = pd.read_parquet(processed_path)

print("Processed data loaded successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Current working directory: c:\Users\John\Desktop\data_sience_group\BigData-Ecommerce-Analysis\notebooks
customer_spending exists: False
top_categories exists: False
monthly_trends exists: False
customer_segments exists: False


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

# 04 Insert into MongoDB

In this step, the merged pandas DataFrame is converted into JSON-like documents and inserted into the MongoDB collection.  
Each document represents one order enriched with customer and payment information.

In [ ]:
collection.delete_many({})

docs = df.to_dict("records")
collection.insert_many(docs)

print("Inserted documents:", collection.count_documents({}))

Inserted documents: 99441


# 05 Create Indexes

Indexes are created to improve query performance.  
The selected fields are commonly used in filtering and lookup operations, such as order ID, customer state, and order status.

In [ ]:
collection.create_index("order_id", unique=True)
collection.create_index("customer_state")
collection.create_index("order_status")

print("Indexes created")

'order_status_1'

# 06 Run Queries

This section contains the main MongoDB queries used in the project.  
The queries are used to count documents, filter orders by state, and aggregate sales by state.

## Query 1: Total Number of Orders

This query counts how many processed order documents are stored in MongoDB.

In [ ]:
print("Total orders:", collection.count_documents({}))

Total orders: 99441


## Query 2: Delivered Orders

This query counts how many orders have the status `delivered`.

In [ ]:
print("Delivered orders:", collection.count_documents({"order_status": "delivered"}))

Delivered orders: 96478


## Query 3: Orders from SP

This query retrieves sample orders from customers in the state of São Paulo.

In [ ]:
results = list(collection.find(
    {"customer_state": "SP"},
    {"_id": 0, "order_id": 1, "customer_city": 1, "customer_state": 1, "payment_value": 1}
).limit(5))

results

[{'order_id': 'e481f51cbdc54678b7cc49136f2d6af7',
  'customer_city': 'sao paulo',
  'payment_value': 38.71},
 {'order_id': 'ad21c59c0840e6cb83a9ceb5573f8159',
  'customer_city': 'santo andre',
  'payment_value': 28.62},
 {'order_id': 'e69bfb5eb88e0ed6a785585b27e16dbf',
  'customer_city': 'sorocaba',
  'payment_value': 169.76},
 {'order_id': '34513ce0c4fab462a55830c0989c7edb',
  'customer_city': 'sao paulo',
  'payment_value': 114.13},
 {'order_id': '5ff96c15d0b717ac6ad1f3d77225a350',
  'customer_city': 'sao paulo',
  'payment_value': 32.7}]

## Query 4: Sales by State

This aggregation query groups the data by customer state and calculates the number of orders and total sales for each state.

In [ ]:
pipeline = [
    {
        "$group": {
            "_id": "$customer_state",
            "orders": {"$sum": 1},
            "total_sales": {"$sum": "$payment_value"}
        }
    },
    {"$sort": {"total_sales": -1}}
]

list(collection.aggregate(pipeline))[:10]

[{'_id': 'SP', 'orders': 41746, 'total_sales': 5998226.96},
 {'_id': 'RJ', 'orders': 12852, 'total_sales': 2144379.69},
 {'_id': 'MG', 'orders': 11635, 'total_sales': 1872257.26},
 {'_id': 'RS', 'orders': 5466, 'total_sales': 890898.54},
 {'_id': 'PR', 'orders': 5045, 'total_sales': 811156.38},
 {'_id': 'SC', 'orders': 3637, 'total_sales': 623086.43},
 {'_id': 'BA', 'orders': 3380, 'total_sales': 616645.82},
 {'_id': 'DF', 'orders': 2140, 'total_sales': 355141.08},
 {'_id': 'GO', 'orders': 2020, 'total_sales': 350092.31},
 {'_id': 'ES', 'orders': 2033, 'total_sales': 325967.55}]

# 07 Performance Test

In this section, we measure the execution time of a filtered MongoDB query.
This gives a simple example of query performance after indexing.

In [ ]:
start = time.time()
list(collection.find({"customer_state": "SP"}))
end = time.time()

print("Query time:", end - start)

Query time: 0.13524436950683594


# Conclusion

In this notebook, the processed parquet output generated by the Spark processing notebook was loaded into MongoDB.
The processed order-level dataset was stored in a MongoDB collection, indexed, and queried for analysis.

The notebook demonstrated total order counting, delivered order filtering, location-based retrieval, state-level sales aggregation, and a simple performance test.
This shows how MongoDB can be used as the storage and query layer of the Spark + MongoDB pipeline.